# Upload your own data

`register()` takes rows you already have in memory. `upload()` takes what is on
disk, which is how data actually arrives: a folder of images with one
subdirectory per class, a CSV someone exported, a YOLO export with its
`data.yaml` beside it.

Nothing about the files is declared. The upload is staged and handed to **the
same prep agent a browser upload goes to**, which reads the layout and the
contents to decide the task, the format and the labels — so a dataset made here
and one made by dragging the folder into the studio are the same dataset.

In [ ]:
import orinth
from pathlib import Path

# Point this at your own data. Anything works: a folder, a single .csv/.jsonl,
# an image directory tree, a Roboflow or COCO export.
SOURCE = Path("~/Downloads/my-dataset").expanduser()

print(SOURCE, "exists:", SOURCE.exists())

If that path does not exist, the next cell writes a small example so the rest of
the notebook runs anyway — sixty rows of text, two classes. Delete it once you
have your own.

In [ ]:
import csv
import tempfile

if not SOURCE.exists():
    SOURCE = Path(tempfile.mkdtemp()) / "example-reviews.csv"
    with SOURCE.open("w", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["text", "label"])
        for index in range(60):
            label = "positive" if index % 2 else "negative"
            opinion = "held up well" if label == "positive" else "stopped working"
            writer.writerow([f"the battery {opinion} after week {index}", label])
    print("using the example instead:", SOURCE)

files = [SOURCE] if SOURCE.is_file() else sorted(p for p in SOURCE.rglob("*") if p.is_file())
print(len(files), "files ·", sum(p.stat().st_size for p in files) / 1e6, "MB")
for path in files[:6]:
    print(" ", path.relative_to(SOURCE.parent))

## Look before applying

`auto_apply=False` stages the files and stops. `detect()` then reports what the
deterministic scan concluded and the evidence behind it — worth reading, because
a wrong `task_type` here becomes a dataset that has to be rebuilt.

In [ ]:
staged = orinth.datasets.upload(SOURCE, name="my data (staged)", auto_apply=False)
detection = orinth.datasets.detect(staged.id)

print("modality  :", detection.get("modality"))
print("task      :", detection.get("task_type"))
print("format    :", detection.get("format"))
print("confidence:", detection.get("confidence"))
print("labels    :", detection.get("candidate_labels"))
print("columns   :", detection.get("columns"))
for signal in detection.get("signals") or []:
    print("  ·", signal)
if detection.get("needs_input"):
    print("\nneeds you:", detection["needs_input"])

## Or just let it run

The default. Stage, detect, plan, split and apply in one call — the same path
the studio takes when you drop a folder on it. It waits, because the id is not
much use until the dataset is real.

In [ ]:
ref = orinth.datasets.upload(SOURCE, name="my data")

print(ref.id)
print(ref.task_type, "·", ref.format)
print(ref.labels)
print(ref.splits)
print(ref.readiness.state, "|", ref.readiness.summary)

## Read it back

If this looks like the data you uploaded, the round trip is done and everything
else in Orinth — training, evaluation, the studio — can now see it.

In [ ]:
frame = orinth.datasets.load(ref.id, "train", limit=5)
print(frame.columns)
frame.head()

### Notes

- **Structure is labelling.** For images, one subdirectory per class is read as
  the class. The relative paths are preserved on upload for exactly that reason.
- **Large folders** are sent in batches of 200 files; nothing has to fit in one
  request.
- **`include=`** narrows the walk: `include=["**/*.jpg", "data.yaml"]`.
- Every upload creates a **new** dataset. Rebuilding one under whatever is
  reading it is the failure this platform already lived through once.